# DARL PPO v3: inicialización conservadora del actor sobre el entorno de DQN v5

**Réplica con semilla del agente 48** (entorno con semilla 42, idéntico al original).

Notebook autónomo para `jeffreyamc/physionet-sepsis-processed`. Semilla 42.
Copia del notebook `kaggle/dqn-dummy/v5/input/main.ipynb` en la que **solo cambia el agente**:
el Double DQN se reemplaza por PPO (actor-crítico con objetivo recortado). Datos, particiones,
deriva, calibración, estado de 30 entradas, acciones A1–A4, recompensa, selección del checkpoint
con B-validation y evaluación congelada en B-test son idénticos a v5. Plan: `plan/ppo-v1.md` (sección v2).

**Cambio frente a PPO v2 (único):** la política inicial prefiere A1 (probabilidad 0,9; 0,033 para A2–A4).
En v2 ningún episodio de entrenamiento terminó sin reentrenar (0 de 1000): el primer A3/A4 degrada el
pipeline de forma irreversible y PPO, que aprende de su propia política, nunca observó la alternativa.
Con la nueva inicialización ≈ 15 % de los episodios iniciales no reentrena (0,933^27).

**Cambios de v2 frente a PPO v1** (v1 casi no movía la política: fracción de recorte ≤ 0,012):
horizonte de 20 episodios (540 decisiones) en lugar de 4, 10 épocas en lugar de 4, minibatch 64,
λ = 0,95, registro de KL aproximada y ratio medio, y sin parada temprana antes del episodio 1000.

## Agente PPO

- Actor 30→128→128→4 (misma arquitectura que la red Q de v5) y crítico 30→128→128→1, sin pesos compartidos.
- Acciones deshabilitadas: logit −∞ antes del softmax.
- Actualización cada 20 episodios con GAE (γ=0,99, λ=0,95), recorte ε=0,2, 10 épocas, minibatch 64,
  Adam 3e-4→3e-5, coeficiente de valor 0,5, entropía 0,01→0,001, norma de gradiente 0,5.
- El crítico aprende retornos estandarizados con media y desviación móviles (normalización de valor).
- Validación y prueba: acción de mayor probabilidad entre las habilitadas (greedy).
- La estabilidad con tolerancia compara logits del actor (tolerancia 0,005), en lugar de valores Q.

## Ejecución y archivos

`RUN_MODE='smoke'`: 50 camas, 8 días B, 6 días A (4+2), 24 episodios.
`RUN_MODE='full'`: protocolo completo y 1000 episodios (sin parada temprana). Las redes son pequeñas; el costo
principal es XGBoost en CPU con dos hilos. Internet deshabilitado en metadata.
Resultados parciales en CSV y figuras PNG; `run_summary.csv` registra modo, versiones, avance y errores.
Una ejecución íntegra puede mostrar que PPO no supera a DDQN ni a las políticas de referencia.


## Configuración reproducible y salidas CSV/PNG

In [ ]:
import os, time, copy, random, hashlib, platform, traceback, itertools
from pathlib import Path
from collections import deque
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier
import xgboost, sklearn
import torch
from torch import nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
AGENT_SEED = 48  # agent randomness only; data, XGBoost and replays keep SEED
RUN_MODE = 'full'  # Change to 'smoke' for the first Kaggle version.
SMOKE = os.environ.get('DARL_DUMMY_SMOKE', '1' if RUN_MODE == 'smoke' else '0') == '1'
VITALS = ['HR','O2Sat','Temp','SBP','MAP','DBP','Resp']
SCENARIOS = ('stable','covariate','concept','combined')
ACTIONS = ('A1','A2','A3','A4')
BEDS, DAYS, A_DAYS, A_FIT_DAYS = (50,8,6,4) if SMOKE else (200,30,30,20)
WARMUP, UPDATE_DAYS, LABEL_DELAY = 3, 3, 1
# v2: sin parada temprana antes de 1000 episodios.
MIN_EPISODES, MAX_EPISODES = (24,24) if SMOKE else (1000,1000)
# PPO: actualización cada ROLLOUT_EPISODES episodios completos.
LR_START, LR_END = 3e-4,3e-5
# v2: configuración de la clase (T_horizon ≈ 540 pasos, K=10, minibatch 64).
GAMMA, GAE_LAMBDA, CLIP_EPS, PPO_EPOCHS, MINIBATCH = .99,.95,.2,10,64
VALUE_COEF, ENT_START, ENT_END, MAX_GRAD_NORM, ROLLOUT_EPISODES = .5,.01,.001,.5,20
# v3: política inicial conservadora (probabilidad de A1 al empezar el entrenamiento).
INIT_A1_PROB = .9
COST_LAMBDA, Q_TIE, SELECT_WINDOW = .05,.005,3
ONSET, CONCEPT_DELAY, CONCEPT_CAP, SCALE_GAIN = 5,2,12.,.4
COV_RATE_GRID, CONCEPT_RATE_GRID = (.25,.35,.5),(.25,.5,1.)
TRAIN_CYCLE = ('stable','covariate','concept','combined','covariate','concept','combined')
OUT = Path(os.environ.get('DARL_DUMMY_OUTPUT', '/kaggle/working' if Path('/kaggle/input').exists() else 'kaggle/ppo-dummy/seeds/ppo-s48/outputs'))
OUT.mkdir(parents=True,exist_ok=True)
(OUT/'figures').mkdir(exist_ok=True)
(OUT/'predictions').mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(AGENT_SEED)
torch.set_num_threads(2)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
progress = dict(version='ppo-v3',init_a1_prob=INIT_A1_PROB,agent='ppo',mode='smoke' if SMOKE else 'full',status='running',stage='setup',seed=SEED,agent_seed=AGENT_SEED,
                days=DAYS,beds=BEDS,a_days=A_DAYS,a_fit_days=A_FIT_DAYS,label_delay_days=LABEL_DELAY,
                update_days=UPDATE_DAYS,device=str(DEVICE),clinical_target='SepsisLabel',
                synthetic_target='scenario_label',xgboost_device='cpu',python=platform.python_version(),
                numpy=np.__version__,pandas=pd.__version__,sklearn=sklearn.__version__,
                xgboost=xgboost.__version__,torch=torch.__version__,architecture='actor 30-128-128-4; critic 30-128-128-1',
                min_episodes=MIN_EPISODES,max_episodes=MAX_EPISODES,gamma=GAMMA,gae_lambda=GAE_LAMBDA,clip_eps=CLIP_EPS,
                ppo_epochs=PPO_EPOCHS,minibatch=MINIBATCH,value_coef=VALUE_COEF,entropy_start=ENT_START,entropy_end=ENT_END,
                max_grad_norm=MAX_GRAD_NORM,rollout_episodes=ROLLOUT_EPISODES,lr_start=LR_START,lr_end=LR_END,
                cost_lambda=COST_LAMBDA,concept_cap_hours=CONCEPT_CAP,drift_onset_day=ONSET,concept_delay_days=CONCEPT_DELAY,
                ks_cuts='0.10,0.30',psi_cuts='0.10,0.25',auc_drop_cuts='0.03,0.10',validation_every=10,
                selection_window=SELECT_WINDOW,kernel_slug='bikaxd/darl-ppo-v3-seed-48')

def checkpoint(stage, **values):
    """Persist progress as CSV after each costly operation."""
    progress.update(stage=stage,**values)
    pd.DataFrame([{'key':k,'value':str(v)} for k,v in progress.items()]).to_csv(OUT/'run_summary.csv',index=False)
    print(f'[{stage}]', {k:v for k,v in values.items() if k in ('episodes_done','status','selected_episode')},flush=True)

def table(name, rows):
    """Persist one inspectable table without JSON outputs."""
    data=rows if isinstance(rows,pd.DataFrame) else pd.DataFrame(rows)
    data.to_csv(OUT/f'{name}.csv',index=False)
    return data

## Carga causal y particiones independientes por paciente

In [ ]:
def find_parquet():
    """Resolve the mounted Kaggle dataset or the authorized local Parquet."""
    explicit=os.environ.get('DARL_DUMMY_PARQUET')
    paths=[Path(explicit)] if explicit else sorted(Path('/kaggle/input').rglob('physionet.parquet'))
    paths.append(Path('data/processed/physionet/physionet.parquet'))
    for path in paths:
        if path.is_file(): return path
    raise FileNotFoundError('physionet.parquet missing; mount jeffreyamc/physionet-sepsis-processed')

def load_frame():
    """Load hourly records and forward fill exclusively within each patient."""
    path=find_parquet()
    frame=pd.read_parquet(path,columns=['patient_id','source_set','ICULOS','SepsisLabel',*VITALS])
    assert not frame.duplicated(['patient_id','ICULOS']).any()
    assert set(frame.source_set.unique())=={'a','b'}
    assert frame.SepsisLabel.isin([0,1]).all()
    frame=frame.sort_values(['patient_id','ICULOS']).reset_index(drop=True)
    assert frame.groupby('patient_id').ICULOS.apply(lambda x:x.is_monotonic_increasing).all()
    frame[VITALS]=frame.groupby('patient_id',sort=False)[VITALS].ffill()
    first_positive=frame.loc[frame.SepsisLabel.eq(1)].groupby('patient_id').ICULOS.min().to_dict()
    patients=frame.groupby(['source_set','patient_id'],sort=False).SepsisLabel.max().reset_index()
    partitions={}
    for site in ('a','b'):
        p=patients.loc[patients.source_set.eq(site)]
        train,rest=train_test_split(p.patient_id.to_numpy(),test_size=.4,stratify=p.SepsisLabel,random_state=SEED)
        val,test=train_test_split(rest,test_size=.5,stratify=p.set_index('patient_id').loc[rest,'SepsisLabel'],random_state=SEED)
        partitions.update({f'{site}_train':sorted(train),f'{site}_validation':sorted(val),f'{site}_test':sorted(test)})
    seen=set()
    for ids in partitions.values():
        assert not seen.intersection(ids)
        seen.update(ids)
    assert len(seen)==frame.patient_id.nunique()
    checkpoint('loaded',data_path=str(path),rows=len(frame),patients=len(seen))
    return frame,partitions,first_positive

def make_replay(frame, ids, days=DAYS, beds=BEDS, rng=None):
    """Consume each patient's hourly sequence once; refill a bed next hour."""
    rng=np.random.default_rng(SEED) if rng is None else rng
    assert days>=1 and beds>0 and len(set(ids))==len(ids)
    histories={pid:g.index.to_numpy() for pid,g in frame.loc[frame.patient_id.isin(ids)].groupby('patient_id',sort=False)}
    queue=list(rng.permutation(sorted(ids)))
    if len(queue)<beds: raise RuntimeError(f'Patient queue exhausted before initialization: {len(queue)} patients, {beds} beds')
    slots=[(queue.pop(),0) for _ in range(beds)]
    indices=[]
    for hour in range(days*24):
        for bed,(pid,pos) in enumerate(slots):
            indices.append(int(histories[pid][pos]))
            pos+=1
            if pos==len(histories[pid]) and hour<days*24-1:
                if not queue:
                    raise RuntimeError(f'Patient queue exhausted at hour {hour}; requested {days} days, {beds} beds, pool={len(ids)}')
                slots[bed]=(queue.pop(),0)
            else: slots[bed]=(pid,pos)
    out=frame.loc[indices].reset_index(drop=True).copy()
    out['hour']=np.repeat(np.arange(days*24),beds)
    out['bed']=np.tile(np.arange(beds),days*24)
    out['day']=out.hour//24+1
    out['label_available_day']=out.day+LABEL_DELAY
    assert len(out)==days*24*beds and out.groupby('hour').size().eq(beds).all()
    assert not out.duplicated(['patient_id','ICULOS']).any()
    assert out.groupby('patient_id').ICULOS.apply(lambda x:x.is_monotonic_increasing).all()
    return out

def inject(schedule, scenario, cov_rate, concept_rate, onset, med, iqr, first_positive):
    """Inject gradual measurement stress and a synthetic positive-label onset shift."""
    assert scenario in SCENARIOS
    out=schedule.copy()
    days=out.day.to_numpy()
    cov=np.maximum(0,days-onset+1)*cov_rate if scenario in ('covariate','combined') else np.zeros(len(out))
    concept_start=onset+(CONCEPT_DELAY if scenario=='combined' else 0)
    shift=np.minimum(CONCEPT_CAP,np.maximum(0,days-concept_start+1)*concept_rate) if scenario in ('concept','combined') else np.zeros(len(out))
    raw=out[VITALS].to_numpy(dtype=np.float32)
    scale=1+SCALE_GAIN*np.minimum(1,cov/3)
    direction=np.array([1,-1,1,1,-1,1,1],dtype=np.float32)
    if scenario in ('covariate','combined'):
        med,iqr=np.asarray(med),np.asarray(iqr)
        shifted=med+scale[:,None]*(raw-med)+cov[:,None]*iqr*direction
        out[VITALS]=np.where(cov[:,None]>0,shifted,raw)
    event=out.patient_id.map(first_positive)
    label=event.notna() & out.ICULOS.ge(event-shift)
    out['scenario_label']=label.astype(np.int8) if scenario in ('concept','combined') else out.SepsisLabel.astype(np.int8)
    out['scenario']=scenario
    out['cov_level']=cov
    out['concept_shift_hours']=shift
    assert out.SepsisLabel.equals(schedule.SepsisLabel)
    assert out.loc[event.isna(),'scenario_label'].eq(0).all()
    assert out.concept_shift_hours.between(0,CONCEPT_CAP).all()
    return out

## Pipeline selectivo, clases desbalanceadas y referencia de monitoreo

In [ ]:
def build_reference(rows):
    """Freeze raw per-vital deciles and a bounded KS sample."""
    result={}
    for col in VITALS:
        clean=rows[col].dropna().to_numpy(dtype=float)
        clean=clean[np.isfinite(clean)]
        if not len(clean): result[col]=None; continue
        edges=np.unique(np.quantile(clean,np.linspace(0,1,11)))
        edges=np.r_[-np.inf,edges[1:-1],np.inf] if len(edges)>2 else np.array([-np.inf,np.inf])
        result[col]=dict(sample=clean[::max(1,len(clean)//1000)][:1000],edges=edges,
                         hist=np.histogram(clean,bins=edges)[0])
    return result

class Pipeline:
    """Robust preprocessing and XGBoost with independently replaceable stages."""
    def __init__(self,med,iqr,model,ref,params,version=0):
        self.med=np.asarray(med,dtype=np.float32).copy()
        self.iqr=np.asarray(iqr,dtype=np.float32).copy()
        self.model,self.ref,self.params,self.version=model,ref,dict(params),version
    def transform(self,rows):
        """Impute with past medians, robust-scale and clip values."""
        raw=rows[VITALS].to_numpy(dtype=np.float32)
        return np.clip((np.where(np.isfinite(raw),raw,self.med)-self.med)/self.iqr,-2,2)
    def predict(self,rows):
        """Return probabilities without consulting any labels."""
        return self.model.predict_proba(self.transform(rows))[:,1]
    def saturation(self,rows):
        """Return the share of feature cells clipped by the current transformer."""
        raw=rows[VITALS].to_numpy(dtype=np.float32)
        return float(np.mean(np.abs((np.where(np.isfinite(raw),raw,self.med)-self.med)/self.iqr)>2))
    def act(self,action,rows):
        """Build a new pipeline using exclusively the supplied labeled past window."""
        assert 0<=action<4
        start=time.perf_counter()
        if action==0:return self,time.perf_counter()-start
        med,iqr=self.med,self.iqr
        if action in (1,3):
            med=rows[VITALS].median().fillna(pd.Series(self.med,index=VITALS)).to_numpy(dtype=np.float32)
            q=rows[VITALS].quantile(.75)-rows[VITALS].quantile(.25)
            iqr=q.fillna(pd.Series(self.iqr,index=VITALS)).clip(lower=1e-6).to_numpy(dtype=np.float32)
        model=self.model
        if action in (2,3):
            y=rows.scenario_label.to_numpy(dtype=np.int8)
            if np.unique(y).size!=2:raise ValueError('Retraining requires both classes in the past window')
            model=XGBClassifier(**self.params,tree_method='hist',device='cpu',n_jobs=2,random_state=SEED,
                                eval_metric='logloss',scale_pos_weight=float((y==0).sum()/max(1,(y==1).sum())))
            model.fit(Pipeline(med,iqr,None,self.ref,self.params).transform(rows),y)
        ref=build_reference(rows) if action in (1,3) else self.ref
        return Pipeline(med,iqr,model,ref,self.params,self.version+1),time.perf_counter()-start

def fit_pipeline(rows,params):
    """Fit both initial stages from an A-train window only."""
    med=rows[VITALS].median().fillna(0.).to_numpy(dtype=np.float32)
    iqr=(rows[VITALS].quantile(.75)-rows[VITALS].quantile(.25)).fillna(1.).clip(lower=1e-6).to_numpy(dtype=np.float32)
    base=Pipeline(med,iqr,None,build_reference(rows),params)
    fitted,_=base.act(2,rows)
    fitted.version=0
    return fitted

def classification_metrics(labels,probabilities):
    """Compute AUC and average precision from stored probabilities."""
    y=np.asarray(labels);p=np.asarray(probabilities)
    valid=np.unique(y).size==2
    return dict(auc=float(roc_auc_score(y,p)) if valid else np.nan,
                average_precision=float(average_precision_score(y,p)) if (y==1).any() else np.nan,
                auc_valid=bool(valid),prevalence=float(np.mean(y)),positives=int((y==1).sum()),rows=len(y))

def monitor_features(rows,pipe):
    """Compute KS/PSI on exactly the daily rows receiving predictions."""
    ks=[];psis=[]
    for col in VITALS:
        current=rows[col].dropna().to_numpy(dtype=float)
        current=current[np.isfinite(current)]
        ref=pipe.ref[col]
        if not len(current) or ref is None:continue
        ks.append(float(ks_2samp(ref['sample'],current,method='asymp').statistic))
        p=ref['hist'].astype(float)+1e-6;q=np.histogram(current,bins=ref['edges'])[0].astype(float)+1e-6
        p/=p.sum();q/=q.sum()
        psis.append(float(np.sum((q-p)*np.log(q/p))))
    return dict(ks=max(ks,default=0.),psi=max(psis,default=0.),monitored_vitals=len(ks),saturation=pipe.saturation(rows))

def encode_observation(metrics,last_auc,auc_reference):
    """Encode KS/PSI and reference-relative AUC deterioration plus validity."""
    valid=np.isfinite(metrics['auc'])
    observed=metrics['auc'] if valid else last_auc
    bins=[np.searchsorted([.10,.30],metrics['ks'],side='right'),
          np.searchsorted([.10,.25],metrics['psi'],side='right'),
          np.searchsorted([.03,.10],round(auc_reference-observed,10),side='right')]
    result=np.zeros(10,dtype=np.float32)
    for i,b in enumerate(bins):result[i*3+b]=1.
    result[9]=float(valid)
    return result

## Ajuste 20+10, selección con A-validation y prueba independiente A-test

In [ ]:
def evaluate_initial(pipe,rows,partition,export=False):
    """Evaluate fixed predictions on A without fitting on evaluation labels."""
    probs=pipe.predict(rows)
    summary=dict(partition=partition,**classification_metrics(rows.scenario_label,probs))
    daily=[]
    for d,indices in rows.groupby('day').groups.items():
        daily.append(dict(partition=partition,day=d,**classification_metrics(rows.loc[indices,'scenario_label'],probs[np.asarray(indices)])))
    if export:
        out=rows[['patient_id','source_set','ICULOS','hour','day','bed','SepsisLabel','scenario_label','label_available_day']].copy()
        out['probability']=probs;out['pipeline_version']=0;out['active_action']='A1';out['policy']='initial';out['scenario']='stable'
        out['prediction_hour']=out.hour;out['label_available_hour']=out.day*24
        out.to_csv(OUT/'predictions/a_test_initial.csv',index=False)
    return summary,daily

def prepare_initial(a_replays):
    """Tune on A-validation; inspect A-test only after choosing the model."""
    train=a_replays['a_train'];first=train.loc[train.day.le(A_FIT_DAYS)].reset_index(drop=True)
    tail=train.loc[train.day.gt(A_FIT_DAYS)].reset_index(drop=True)
    results=[];best=None;winner=None;rank=None
    for i,(depth,lr,trees) in enumerate(itertools.product((3,5),(.05,.1),(60,120))):
        params=dict(max_depth=depth,learning_rate=lr,n_estimators=trees)
        early=fit_pipeline(first,params)
        temporal,_=evaluate_initial(early,tail,'a_train_future')
        full=fit_pipeline(train,params)
        val,_=evaluate_initial(full,a_replays['a_validation'],'a_validation')
        if not np.isfinite(val['auc']):raise ValueError('A-validation needs both classes to select hyperparameters')
        row=dict(candidate=i,**params,temporal_auc=temporal['auc'],temporal_ap=temporal['average_precision'],
                 validation_auc=val['auc'],validation_ap=val['average_precision'])
        results.append(row);table('xgboost_search',results)
        candidate=(val['auc'],val['average_precision'],-trees,-depth,-lr)
        if rank is None or candidate>rank:best,winner,rank=full,row,candidate
    summary=[];days=[]
    for part in ('a_train','a_validation','a_test'):
        info,daily=evaluate_initial(best,a_replays[part],part,export=part=='a_test')
        summary.append(info);days.extend(daily)
    table('a_metrics',summary);daily=table('a_daily_metrics',days)
    auc_reference=float(daily.loc[daily.partition.eq('a_validation'),'auc'].mean())
    assert np.isfinite(auc_reference)
    checkpoint('initial_model',selected_xgboost_candidate=winner['candidate'],xgboost_params=str(best.params),
               auc_reference=auc_reference,a_test_auc=summary[-1]['auc'])
    return best,auc_reference

## PPO actor-crítico con máscara de acciones y objetivo recortado

In [ ]:
class QNetwork(nn.Module):
    """Actor head: four action logits from the last three daily observation blocks (same shape as v5)."""
    def __init__(self,outputs=4):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(30,128),nn.ReLU(),nn.Linear(128,128),nn.ReLU(),nn.Linear(128,outputs))
    def forward(self,x):
        """Return unconstrained logits (or one state value for the critic)."""
        return self.net(x)

class PPOAgent:
    """Clipped PPO; only training episodes add transitions or update weights."""
    def __init__(self):
        self.online=QNetwork().to(DEVICE)          # actor; kept as `online` for checkpoints and hashes
        # Conservative start: near state-independent logits with P(A1)=INIT_A1_PROB, rest uniform.
        with torch.no_grad():
            last=self.online.net[-1];last.weight.mul_(.01)
            bias=np.log(INIT_A1_PROB*(len(ACTIONS)-1)/(1-INIT_A1_PROB))
            last.bias.copy_(torch.tensor([bias,0.,0.,0.],dtype=torch.float32,device=DEVICE))
        self.critic=QNetwork(outputs=1).to(DEVICE)
        self.optim=torch.optim.Adam(list(self.online.parameters())+list(self.critic.parameters()),lr=LR_START)
        self.rng=np.random.default_rng(AGENT_SEED);self.torch_rng=torch.Generator(device='cpu').manual_seed(AGENT_SEED)
        self.entropy_coef=ENT_START;self.updates=0;self.losses=[];self.stats={}
        self.rollout=[];self.episode=[];self.pending=None;self.episodes_in_rollout=0
        # Running return statistics: the critic learns standardized returns (value normalization).
        self.ret_count,self.ret_mean,self.ret_m2=1e-4,0.,1.
    def q_values(self,s):
        """Return actor logits without gradients (stored as q_* columns for v5 compatibility)."""
        with torch.no_grad():
            x=torch.as_tensor(s,dtype=torch.float32,device=DEVICE).unsqueeze(0)
            return self.online(x).squeeze(0).cpu().numpy()
    def select(self,s,mask,greedy=False):
        """Sample during training; most probable feasible action in validation/test."""
        logits=np.where(mask,self.q_values(s),-np.inf)
        if greedy:return int(np.argmax(logits))
        probs=np.exp(logits-logits.max());probs/=probs.sum()
        action=int(self.rng.choice(4,p=probs))
        with torch.no_grad():
            value=float(self.critic(torch.as_tensor(s,dtype=torch.float32,device=DEVICE).unsqueeze(0)).item())*self.ret_std+self.ret_mean
        self.pending=dict(s=s.copy(),mask=np.asarray(mask,dtype=bool).copy(),a=action,
                          logp=float(np.log(probs[action])),v=value)
        return action
    def remember(self,reward,done):
        """Attach the delayed reward to the last sampled decision."""
        assert self.pending is not None and np.isfinite(reward)
        self.episode.append(dict(self.pending,r=float(reward),done=bool(done)));self.pending=None
    @property
    def ret_std(self):
        """Return the running standard deviation of observed returns."""
        return float(np.sqrt(max(self.ret_m2/self.ret_count,1e-8)))
    def _track_returns(self,values):
        """Update running return mean/variance with a batch (parallel Welford)."""
        values=np.asarray(values,dtype=np.float64);n=len(values)
        mean,var=values.mean(),values.var();delta=mean-self.ret_mean;total=self.ret_count+n
        self.ret_mean+=delta*n/total
        self.ret_m2+=var*n+delta**2*self.ret_count*n/total
        self.ret_count=total
    def _advantages(self,steps):
        """Generalized advantage estimation inside one complete episode."""
        adv=np.zeros(len(steps),dtype=np.float32);last=0.
        for t in reversed(range(len(steps))):
            following=0. if steps[t]['done'] else steps[t+1]['v']
            delta=steps[t]['r']+GAMMA*following-steps[t]['v']
            last=delta+GAMMA*GAE_LAMBDA*(0. if steps[t]['done'] else 1.)*last
            adv[t]=last
        return adv
    def end_episode(self,fraction):
        """Close the episode, update every ROLLOUT_EPISODES and decay lr/entropy."""
        if self.episode:
            adv=self._advantages(self.episode)
            for step,a in zip(self.episode,adv):step['adv']=float(a);step['ret']=float(a+step['v'])
            self.rollout.extend(self.episode);self.episode=[];self.episodes_in_rollout+=1
        if self.episodes_in_rollout>=ROLLOUT_EPISODES:self._update()
        f=min(1.,fraction)
        for g in self.optim.param_groups:g['lr']=LR_START+(LR_END-LR_START)*f
        self.entropy_coef=ENT_START+(ENT_END-ENT_START)*f
    def _update(self):
        """Clipped surrogate, value loss and entropy bonus over the collected rollout."""
        data=self.rollout;self.rollout=[];self.episodes_in_rollout=0
        self._track_returns([d['ret'] for d in data])
        t=lambda key,dtype=torch.float32:torch.as_tensor(np.array([d[key] for d in data]),dtype=dtype,device=DEVICE)
        states,masks,actions=t('s'),t('mask',torch.bool),t('a',torch.long)
        old_logp,returns,adv=t('logp'),t('ret'),t('adv')
        adv=(adv-adv.mean())/(adv.std()+1e-8)
        returns=(returns-self.ret_mean)/self.ret_std
        stats=[]
        for _ in range(PPO_EPOCHS):
            order=torch.randperm(len(data),generator=self.torch_rng).to(DEVICE)
            for start in range(0,len(data),MINIBATCH):
                idx=order[start:start+MINIBATCH]
                # Finite mask value: zero probability without NaN gradients in the entropy term.
                logits=self.online(states[idx]).masked_fill(~masks[idx],-1e9)
                dist=torch.distributions.Categorical(logits=logits)
                logp=dist.log_prob(actions[idx]);ratio=torch.exp(logp-old_logp[idx])
                policy_loss=-torch.min(ratio*adv[idx],torch.clamp(ratio,1-CLIP_EPS,1+CLIP_EPS)*adv[idx]).mean()
                value_loss=nn.functional.mse_loss(self.critic(states[idx]).squeeze(1),returns[idx])
                entropy=dist.entropy().mean()
                loss=policy_loss+VALUE_COEF*value_loss-self.entropy_coef*entropy
                self.optim.zero_grad();loss.backward()
                torch.nn.utils.clip_grad_norm_(list(self.online.parameters())+list(self.critic.parameters()),MAX_GRAD_NORM)
                self.optim.step();self.updates+=1
                with torch.no_grad():
                    log_ratio=logp-old_logp[idx]
                    approx_kl=float(((ratio-1)-log_ratio).mean().item())  # estimador k3 de KL(old||new)
                stats.append((float(policy_loss.item()),float(value_loss.item()),float(entropy.item()),
                              float(((ratio-1).abs()>CLIP_EPS).float().mean().item()),approx_kl,
                              float(ratio.mean().item()),float(ratio.min().item()),float(ratio.max().item())))
        s=np.mean(stats,axis=0);arr=np.asarray(stats)
        self.stats=dict(policy_loss=s[0],value_loss=s[1],entropy=s[2],clip_fraction=s[3],approx_kl=s[4],
                        ratio_mean=s[5],ratio_min=float(arr[:,6].min()),ratio_max=float(arr[:,7].max()),
                        rollout_steps=len(data))
        self.losses.append(float(s[0]+VALUE_COEF*s[1]))
    @property
    def lr(self):
        """Return the current learning rate."""
        return self.optim.param_groups[0]['lr']
    def fingerprint(self):
        """Hash actor and critic weights to prove validation/test do not train the agent."""
        h=hashlib.sha256()
        for net in (self.online,self.critic):
            for tensor in net.state_dict().values():h.update(tensor.detach().cpu().numpy().tobytes())
        return h.hexdigest()


## Entorno diario: predecir primero, recibir etiquetas al día siguiente

In [ ]:
class DailyEnv:
    """One patient stream with frozen predictions and one-day label latency."""
    def __init__(self,schedule,initial,auc_reference,cost_unit,prediction_path=None,policy='ppo'):
        self.schedule=schedule
        self.parts={int(d):r.reset_index(drop=True) for d,r in schedule.groupby('day',sort=True)}
        self.days=int(schedule.day.max());assert self.days>WARMUP
        self.pipe=initial;self.auc_reference=auc_reference;self.last_auc=auc_reference
        self.cost_unit=max(cost_unit,1e-6);self.prediction_path=prediction_path;self.policy=policy
        self.history=deque(maxlen=3);self.daily=[];self.actions=[];self.pending_probabilities=None
        self.day=1
        for d in range(1,WARMUP+1):
            self.day=d
            pending=self._predict(d,0,0.,None)
            self.day=d+1
            self._release(pending,active=False)
    def state(self):
        """Return only released observation blocks before the decision day."""
        assert len(self.history)==3 and self.daily[-1]['day']==self.day-1
        return np.concatenate(list(self.history)).astype(np.float32)
    def training_rows(self):
        """Return the last three days whose labels have already arrived."""
        selected=[self.parts[d] for d in range(max(1,self.day-UPDATE_DAYS),min(self.day,self.days+1))]
        out=pd.concat(selected,ignore_index=True)
        assert out.day.max()<self.day and out.label_available_day.max()<=self.day
        return out
    def action_mask(self):
        """Disable classifier fits when the past labeled window has one class."""
        rows=self.training_rows();both=rows.scenario_label.nunique()==2
        return np.array([True,len(rows)>0,both,both],dtype=bool)
    def _predict(self,day,action,elapsed,state):
        rows=self.parts[day]
        # No within-day fits: vectorized prediction equals the hourly stream with this frozen pipeline.
        probability=np.array(self.pipe.predict(rows),copy=True)
        self.pending_probabilities=probability.copy()
        metrics=monitor_features(rows,self.pipe)
        return dict(day=day,rows=rows,probability=probability,metrics=metrics,action=action,
                    elapsed=float(elapsed),version=self.pipe.version,state=state)
    def _release(self,pending,active):
        day=pending['day'];rows=pending['rows'];probability=pending['probability']
        assert self.day==day+LABEL_DELAY and rows.label_available_day.le(self.day).all()
        observed=classification_metrics(rows.scenario_label,probability)
        original=classification_metrics(rows.SepsisLabel,probability)
        cost=COST_LAMBDA*pending['elapsed']/self.cost_unit
        reward=float((observed['auc'] if observed['auc_valid'] else 0.)-cost) if active else np.nan
        record=dict(day=day,policy=self.policy,scenario=rows.scenario.iloc[0],**pending['metrics'],**observed,
                    auc_original=original['auc'],average_precision_original=original['average_precision'],
                    auc_drop=self.auc_reference-observed['auc'],action=ACTIONS[pending['action']],
                    reward=reward,cost=cost,time_s=pending['elapsed'],pipeline_version=pending['version'],
                    label_received_day=self.day,warmup=not active,patients=int(rows.patient_id.nunique()),
                    cov_level=float(rows.cov_level.iloc[0]),concept_shift_hours=float(rows.concept_shift_hours.iloc[0]))
        self.daily.append(record)
        self.history.append(encode_observation(record,self.last_auc,self.auc_reference))
        if observed['auc_valid']:self.last_auc=observed['auc']
        if self.prediction_path is not None:
            out=rows[['patient_id','source_set','ICULOS','hour','day','bed','SepsisLabel','scenario_label','label_available_day']].copy()
            out['probability']=probability;out['pipeline_version']=pending['version'];out['active_action']=record['action']
            out['prediction_hour']=out.hour;out['label_available_hour']=out.day*24
            out['policy']=self.policy;out['scenario']=record['scenario']
            path=Path(self.prediction_path)
            out.to_csv(path,mode='a',header=not path.exists(),index=False)
        return record
    def oracle(self):
        """Audit feasible alternatives offline; no future labels reach the controller."""
        before=self.pipe;past=self.training_rows();today=self.parts[self.day];mask=self.action_mask()
        utilities=[];values=[]
        for action in range(4):
            if not mask[action]:utilities.append(np.nan);values.append(np.nan);continue
            alternative,seconds=before.act(action,past)
            value=classification_metrics(today.scenario_label,alternative.predict(today))['auc']
            values.append(value)
            utilities.append((value if np.isfinite(value) else 0.)-COST_LAMBDA*seconds/self.cost_unit)
        assert self.pipe is before
        return utilities,values
    def step(self,requested):
        """Act using known labels, predict this day, and close its delayed reward."""
        if self.day>self.days:raise RuntimeError('Episode already finished')
        d=self.day;state=self.state();past=self.training_rows();mask=self.action_mask()
        requested=int(requested);executed=requested if mask[requested] else 0
        self.pipe,seconds=self.pipe.act(executed,past)
        pending=self._predict(d,executed,seconds,state.copy())
        self.day=d+LABEL_DELAY
        result=self._release(pending,active=True)
        done=d==self.days
        info=dict(result,requested=requested,executed=executed,state_end_day=d-1,
                  training_min_day=int(past.day.min()),training_max_day=int(past.day.max()),
                  training_label_available_max=int(past.label_available_day.max()),
                  fallback_reason='' if requested==executed else 'one_class_past_window',
                  **{f'mask_{name}':bool(mask[i]) for i,name in enumerate(ACTIONS)})
        self.actions.append(info)
        assert info['training_label_available_max']<=d
        return self.state(),result['reward'],done,info

def choose_policy(policy,agent,state,mask,metrics,rng,auc_reference):
    """Select a feasible action; fixed baselines record explicit A1 fallbacks."""
    if policy=='ppo':return agent.select(state,mask,greedy=True)
    if policy in ACTIONS:return ACTIONS.index(policy)
    if policy=='random':return int(rng.choice(np.flatnonzero(mask)))
    cov=metrics['ks']>=.30 or metrics['psi']>=.25
    low=np.isfinite(metrics['auc']) and auc_reference-metrics['auc']>=.03
    requested=3 if cov and low else 1 if cov else 2 if low else 0
    return requested if mask[requested] else 0

def roll(schedule,policy,initial,auc_reference,cost_unit,agent=None,audit=False,export=False):
    """Evaluate a frozen controller and retain daily metrics and counterfactual audit."""
    path=OUT/'predictions'/f'b_test_{schedule.scenario.iloc[0]}_{policy}.csv' if export else None
    if path is not None and path.exists():raise FileExistsError(f'Refusing mixed prediction runs: {path}')
    before_hash=agent.fingerprint() if agent is not None else None
    env=DailyEnv(schedule,initial,auc_reference,cost_unit,path,policy)
    rng=np.random.default_rng(SEED);audits=[]
    while env.day<=env.days:
        s=env.state();mask=env.action_mask()
        action=choose_policy(policy,agent,s,mask,env.daily[-1],rng,auc_reference)
        qs=agent.q_values(s) if agent is not None and policy=='ppo' else np.full(4,np.nan)
        alternatives,values=env.oracle() if audit else (None,None)
        _,_,done,info=env.step(action)
        info.update({f'q_{name}':float(qs[i]) for i,name in enumerate(ACTIONS)})
        info.update({f'state_{i}':float(value) for i,value in enumerate(s)})
        if audit:
            alternatives[info['executed']]=info['reward']
            winner=int(np.nanargmax(alternatives))
            audits.append(dict(scenario=info['scenario'],day=info['day'],chosen=ACTIONS[info['executed']],
                               oracle=ACTIONS[winner],oracle_action=winner,oracle_utility=float(alternatives[winner]),chosen_utility=info['reward'],
                               regret=float(alternatives[winner]-info['reward']),
                               **{f'utility_{name}':alternatives[i] for i,name in enumerate(ACTIONS)},
                               **{f'auc_{name}':values[i] for i,name in enumerate(ACTIONS)}))
        if done:break
    if agent is not None:assert agent.fingerprint()==before_hash
    daily=pd.DataFrame(env.daily);actions=pd.DataFrame(env.actions)
    result=dict(scenario=schedule.scenario.iloc[0],policy=policy,mean_auc=float(daily.auc.mean()),
                mean_auc_original=float(daily.auc_original.mean()),mean_ap=float(daily.average_precision.mean()),
                **{'return':float(actions.reward.sum())},cost=float(actions.cost.sum()),seconds=float(actions.time_s.sum()),
                valid_days=int(daily.auc_valid.sum()),decisions=len(actions),patients=int(schedule.patient_id.nunique()),
                **{f'count_{name}':int(actions.executed.eq(i).sum()) for i,name in enumerate(ACTIONS)})
    assert len(actions)==DAYS-WARMUP and daily.day.max()==DAYS
    return result,actions,daily,pd.DataFrame(audits)

## Calibración solo en B-train y unidad de costo congelada

In [ ]:
def calibrate(base,initial,auc_reference,first_positive):
    """Choose slow, measurable stress using B-train exclusively; report unmet gates."""
    stable=inject(base,'stable',0,0,ONSET,initial.med,initial.iqr,first_positive)
    history=stable.loc[stable.day.between(1,3)]
    if history.scenario_label.nunique()!=2:raise ValueError('B-train cost window lacks both classes')
    timings=[initial.act(3,history)[1] for _ in range(5)]
    cost_unit=max(float(np.median(timings)),1e-6)
    table('cost_calibration',[dict(repetition=i+1,a4_seconds=v,cost_unit=cost_unit) for i,v in enumerate(timings)])
    candidates=[];rates={}
    for kind,grid,repair in (('covariate',COV_RATE_GRID,1),('concept',CONCEPT_RATE_GRID,2)):
        for rate in grid:
            scenario=inject(base,kind,rate if kind=='covariate' else 0.,rate if kind=='concept' else 0.,
                            ONSET,initial.med,initial.iqr,first_positive)
            first=min(ONSET,DAYS);last=DAYS
            def value(rows,pipe):return classification_metrics(rows.scenario_label,pipe.predict(rows))['auc']
            early0=value(stable.loc[stable.day.eq(first)],initial);late0=value(stable.loc[stable.day.eq(last)],initial)
            early=value(scenario.loc[scenario.day.eq(first)],initial);late=value(scenario.loc[scenario.day.eq(last)],initial)
            past=scenario.loc[scenario.day.between(last-3,last-1)]
            changed,_=initial.act(repair,past) if repair==1 or past.scenario_label.nunique()==2 else (initial,0.)
            recovered=value(scenario.loc[scenario.day.eq(last)],changed)
            qualified=bool(np.isfinite([early0,late0,early,late,recovered]).all() and early0-early<=.05 and late0-late>=.10 and recovered-late>=.05)
            candidates.append(dict(kind=kind,rate=rate,early_drop=early0-early,late_drop=late0-late,recovery=recovered-late,qualified=qualified))
        subset=pd.DataFrame([r for r in candidates if r['kind']==kind])
        qualified=subset.loc[subset.qualified]
        chosen=(qualified.sort_values('rate') if len(qualified) else subset.sort_values(['late_drop','rate'],ascending=[False,True],na_position='last')).iloc[0]
        rates[kind]=float(chosen.rate)
    report=table('calibration',candidates)
    calibrated=bool(report.groupby('kind').qualified.any().all())
    checkpoint('calibrated',cov_rate=rates['covariate'],concept_rate_hours=rates['concept'],cost_unit_s=cost_unit,calibration_qualified=calibrated)
    return rates['covariate'],rates['concept'],cost_unit,calibrated

## Entrenamiento con B-train y selección de checkpoint con B-validation

In [ ]:
def masked_tie_agreement(reference_q,candidate_q,masks,tolerance=Q_TIE):
    """Compare feasible greedy choices on identical probe states and masks."""
    reference=np.argmax(np.where(masks,reference_q,-np.inf),axis=1)
    candidate=np.argmax(np.where(masks,candidate_q,-np.inf),axis=1)
    rows=np.arange(len(reference))
    delta=np.abs(reference_q[rows,reference]-reference_q[rows,candidate])
    return float(np.mean((reference==candidate)|(delta<=tolerance)))

def train_agent(train_bases,val_base,initial,auc_reference,cost_unit,cov_rate,concept_rate,first_positive):
    """Learn in B-train and choose one checkpoint without consulting B-test."""
    agent=PPOAgent();rng=np.random.default_rng(AGENT_SEED);history=[];checkpoints=[]
    vals=[inject(val_base,s,cov_rate,concept_rate,ONSET,initial.med,initial.iqr,first_positive) for s in SCENARIOS]
    for ep in range(1,MAX_EPISODES+1):
        kind=TRAIN_CYCLE[(ep-1)%len(TRAIN_CYCLE)]
        onset=int(rng.integers(3,8)) if not SMOKE else ONSET
        factor=float(rng.choice((.75,1.,1.25)))
        schedule=inject(train_bases[(ep-1)%len(train_bases)],kind,cov_rate*factor,concept_rate*factor,onset,initial.med,initial.iqr,first_positive)
        env=DailyEnv(schedule,initial,auc_reference,cost_unit)
        total=0.;counts=np.zeros(4,dtype=int)
        entropy_coef,lr=agent.entropy_coef,agent.lr
        while env.day<=env.days:
            s=env.state();action=agent.select(s,env.action_mask())
            s2,reward,done,info=env.step(action)
            agent.remember(reward,done)
            total+=reward;counts[info['executed']]+=1
            if done:break
        row=dict(episode=ep,scenario=kind,onset=onset,rate_factor=factor,train_return=total,entropy_coef=entropy_coef,lr=lr,
                 loss=agent.losses[-1] if agent.losses else np.nan,updates=agent.updates,validation_return=np.nan,
                 **{f'train_count_{name}':int(counts[i]) for i,name in enumerate(ACTIONS)})
        agent.end_episode(ep/MAX_EPISODES)
        if agent.stats.get('_episode')!=agent.updates:
            row.update({k:v for k,v in agent.stats.items() if not k.startswith('_')});agent.stats['_episode']=agent.updates
        if ep%10==0 or ep==MAX_EPISODES:
            scores=[];signature=[];q=[];probe_states=[];probe_masks=[];vc=np.zeros(4,dtype=int)
            for schedule in vals:
                result,actions,_,_=roll(schedule,'ppo',initial,auc_reference,cost_unit,agent)
                scores.append(result['return']);signature.extend(actions.executed.tolist())
                q.append(actions[[f'q_{name}' for name in ACTIONS]].to_numpy())
                probe_states.append(actions[[f'state_{i}' for i in range(30)]].to_numpy(dtype=np.float32))
                probe_masks.append(actions[[f'mask_{name}' for name in ACTIONS]].to_numpy(dtype=bool))
                vc+=np.bincount(actions.executed,minlength=4)
            row['validation_return']=float(np.mean(scores))
            row.update({f'val_count_{name}':int(vc[i]) for i,name in enumerate(ACTIONS)})
            signature=np.asarray(signature)
            current=dict(episode=ep,score=row['validation_return'],signature=signature,q=np.concatenate(q),
                         probe_states=np.concatenate(probe_states),probe_masks=np.concatenate(probe_masks),
                         weights=copy.deepcopy(agent.online.state_dict()))
            if checkpoints:row['agreement_prev']=float(np.mean(signature==checkpoints[-1]['signature']))
            checkpoints.append(current)
            table('training_history',history+[row])
            checkpoint('training',episodes_done=ep,updates=agent.updates,best_validation_return=max(c['score'] for c in checkpoints))
        history.append(row)
        if ep>=MIN_EPISODES and len(checkpoints)>=5:
            latest=checkpoints[-5:];scores=np.array([c['score'] for c in latest])
            spread=float(np.ptp(scores)/max(abs(scores.mean()),1e-6))
            agreements=[float(np.mean(a['signature']==b['signature'])) for a,b in zip(latest,latest[1:])]
            if spread<=.05 and min(agreements)>=.95:break
    smooth=pd.Series([c['score'] for c in checkpoints]).rolling(SELECT_WINDOW,min_periods=SELECT_WINDOW).mean()
    index=int(smooth.idxmax());winner=checkpoints[index]
    agent.online.load_state_dict(winner['weights'])
    selected=checkpoints[max(0,index-SELECT_WINDOW+1):index+1]
    scores=np.array([c['score'] for c in selected])
    agreements=[float(np.mean(c['signature']==winner['signature'])) for c in selected[:-1]]
    ties=[];probe=QNetwork().to(DEVICE)
    states=torch.as_tensor(winner['probe_states'],device=DEVICE)
    for c in selected[:-1]:
        probe.load_state_dict(c['weights'])
        with torch.no_grad():candidate_q=probe(states).cpu().numpy()
        ties.append(masked_tie_agreement(winner['q'],candidate_q,winner['probe_masks']))
    training=dict(selected_episode=winner['episode'],selected_validation_return=winner['score'],
                  selected_smoothed_return=float(smooth.iloc[index]),episodes=len(history),updates=agent.updates,
                  selected_window_variation=float(np.ptp(scores)/max(abs(scores.mean()),1e-6)),
                  selected_window_agreement=min(agreements) if agreements else np.nan,
                  selected_window_tie_agreement=min(ties) if ties else np.nan,
                  tie_probe='selected_checkpoint_validation_states_and_masks',
                  convergence_assessable=len(selected)==SELECT_WINDOW)
    table('training_history',history);table('checkpoint_selection',[training])
    checkpoint('trained',**training)
    return agent,pd.DataFrame(history),training

## Figuras PNG y auditoría de resultados guardados

In [ ]:
def figure(name):
    """Save the active figure exclusively as PNG."""
    plt.tight_layout();plt.savefig(OUT/'figures'/f'{name}.png',dpi=160);plt.close()

def render_figures(a_replays,b_replays,history,daily,actions,comparisons,audits):
    """Render temporal preparation, monitoring and policy comparisons."""
    search=pd.read_csv(OUT/'xgboost_search.csv')
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for name,rows in a_replays.items():
        axes[0].plot(rows.groupby('day').size(),label=name)
    axes[0].axvline(A_FIT_DAYS+.5,color='black',ls='--');axes[0].set(title='A: entrenamiento temporal y comprobación',xlabel='Día simulado',ylabel='Registros');axes[0].legend()
    axes[1].plot(np.arange(len(search))+1,search.validation_auc,marker='o',label='A-validation')
    axes[1].plot(np.arange(len(search))+1,search.temporal_auc,marker='o',label='A-train: tramo futuro')
    axes[1].set(title='Ocho configuraciones de XGBoost',xlabel='Configuración',ylabel='AUC');axes[1].legend();figure('a_preparation')
    fig,axes=plt.subplots(2,1,figsize=(10,6))
    for name,rows in b_replays.items():
        axes[0].plot(rows.groupby('hour').size(),label=name)
        first=rows.drop_duplicates('patient_id');axes[1].plot(first.groupby('day').size(),label=name)
    axes[0].set(ylabel='Camas ocupadas',xlabel='Hora simulada');axes[0].legend()
    axes[1].set(ylabel='Pacientes nuevos',xlabel='Día simulado');axes[1].legend();figure('occupancy')
    dqn=daily.loc[daily.policy.eq('ppo')]
    fig,axes=plt.subplots(2,1,figsize=(10,6))
    for scenario,g in dqn.groupby('scenario'):
        axes[0].plot(g.day,g.cov_level,label=scenario);axes[1].plot(g.day,g.concept_shift_hours,label=scenario)
    axes[0].set(ylabel='Desplazamiento (IQR)',title='Calendario de deriva artificial');axes[0].legend()
    axes[1].set(ylabel='Adelanto adicional (horas)',xlabel='Día simulado');axes[1].legend();figure('drift_calendar')
    fig,axes=plt.subplots(3,1,figsize=(11,8),sharex=True)
    g=dqn.loc[dqn.scenario.eq('combined')]
    axes[0].plot(g.day,g.ks,label='KS');axes[0].plot(g.day,g.psi,label='PSI');axes[0].legend()
    axes[1].plot(g.day,g.auc,label='AUC scenario_label');axes[1].plot(g.day,g.auc_original,label='AUC SepsisLabel');axes[1].legend()
    axes[2].plot(g.day,g.pipeline_version,label='Versión del pipeline');axes[2].set(xlabel='Día simulado');axes[2].legend();figure('monitoring')
    fig,axes=plt.subplots(3,1,figsize=(11,8))
    axes[0].plot(history.episode,history.train_return,label='Retorno entrenamiento')
    valid=history.dropna(subset=['validation_return']);axes[0].plot(valid.episode,valid.validation_return,label='Validación greedy');axes[0].legend()
    axes[1].plot(history.episode,history.entropy_coef,label='Coeficiente de entropía')
    if 'entropy' in history:axes[1].plot(history.episode,history.entropy,label='Entropía de la política')
    axes[1].legend()
    axes[2].plot(history.episode,history.loss,label='Pérdida PPO (actor + 0,5·crítico)');axes[2].set(xlabel='Episodio');axes[2].legend();figure('training')
    fig,axes=plt.subplots(2,2,figsize=(13,8))
    for ax,scenario in zip(axes.flat,SCENARIOS):
        g=comparisons.loc[comparisons.scenario.eq(scenario)]
        ax.bar(g.policy,g.mean_auc);ax.set(title=scenario,ylabel='AUC diario medio',ylim=(0,1));ax.tick_params(axis='x',rotation=35)
    figure('auc_by_policy')
    fig,axes=plt.subplots(2,2,figsize=(13,8))
    for ax,scenario in zip(axes.flat,SCENARIOS):
        g=comparisons.loc[comparisons.scenario.eq(scenario)]
        ax.bar(g.policy,g['return']);ax.set(title=scenario,ylabel='Retorno acumulado');ax.tick_params(axis='x',rotation=35)
    figure('return_by_policy')
    fig,axes=plt.subplots(2,2,figsize=(12,7))
    for ax,scenario in zip(axes.flat,SCENARIOS):
        g=actions.loc[actions.scenario.eq(scenario)&actions.policy.eq('ppo')]
        ax.step(g.day,g.executed+1,where='mid',label='PPO')
        o=audits.loc[audits.scenario.eq(scenario)];ax.step(o.day,o.oracle_action+1,where='mid',label='Oracle offline',alpha=.7)
        ax.set(title=scenario,xlabel='Día',yticks=[1,2,3,4],yticklabels=ACTIONS);ax.legend()
    figure('actions')
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for scenario,g in audits.groupby('scenario'):
        axes[0].plot(g.day,g.regret.cumsum(),label=scenario)
    axes[0].set(title='Regret acumulado frente al oracle inmediato',xlabel='Día',ylabel='Regret');axes[0].legend()
    g=comparisons.groupby('policy').seconds.sum();axes[1].bar(g.index,g.values);axes[1].set(title='Costo de actualización, cuatro escenarios',ylabel='Segundos');axes[1].tick_params(axis='x',rotation=35)
    figure('regret_and_cost')
    if 'approx_kl' in history:
        upd=history.dropna(subset=['approx_kl'])
        fig,axes=plt.subplots(3,1,figsize=(10,8),sharex=True)
        axes[0].plot(upd.episode,upd.ratio_mean,label='Ratio medio');axes[0].fill_between(upd.episode,upd.ratio_min,upd.ratio_max,alpha=.2,label='Ratio mín–máx')
        axes[0].axhline(1-CLIP_EPS,ls='--',color='gray');axes[0].axhline(1+CLIP_EPS,ls='--',color='gray');axes[0].set(ylabel='r_t(θ)');axes[0].legend()
        axes[1].plot(upd.episode,upd.clip_fraction,label='Fracción recortada');axes[1].set(ylabel='Fracción');axes[1].legend()
        axes[2].plot(upd.episode,upd.approx_kl,label='KL aproximada');axes[2].set(ylabel='KL',xlabel='Episodio');axes[2].legend()
        figure('ppo_diagnostics')

def verify_exports(daily,actions,initial,agent,frame):
    """Reconstruct daily AUC and check causal CSV provenance for every policy."""
    checks=[]
    original=frame.set_index(['patient_id','ICULOS']).SepsisLabel
    for path in sorted((OUT/'predictions').glob('b_test_*.csv')):
        rows=pd.read_csv(path)
        assert len(rows)==DAYS*24*BEDS and not rows.duplicated(['patient_id','ICULOS']).any()
        assert rows.groupby('hour').size().eq(BEDS).all()
        assert rows.groupby('patient_id').ICULOS.apply(lambda s:s.is_monotonic_increasing).all()
        assert (rows.label_available_day==rows.day+1).all()
        assert np.array_equal(rows.SepsisLabel,original.reindex(pd.MultiIndex.from_frame(rows[['patient_id','ICULOS']])).to_numpy())
        reference=daily.loc[daily.scenario.eq(rows.scenario.iloc[0])&daily.policy.eq(rows.policy.iloc[0])].set_index('day')
        for day,g in rows.groupby('day'):
            for label,key in [('scenario_label','auc'),('SepsisLabel','auc_original')]:
                value=classification_metrics(g[label],g.probability)['auc'];saved=reference.loc[day,key]
                assert (np.isnan(value) and np.isnan(saved)) or abs(value-saved)<1e-9
        checks.append(dict(check='saved_predictions_auc_and_stream',artifact=path.name,passed=True,rows=len(rows)))
    rows=pd.read_csv(OUT/'predictions/a_test_initial.csv')
    a_daily=pd.read_csv(OUT/'a_daily_metrics.csv');a_daily=a_daily.loc[a_daily.partition.eq('a_test')].set_index('day')
    for day,g in rows.groupby('day'):
        value=classification_metrics(g.SepsisLabel,g.probability)['auc'];saved=a_daily.loc[day,'auc']
        assert (np.isnan(value) and np.isnan(saved)) or abs(value-saved)<1e-9
    assert np.array_equal(rows.SepsisLabel,original.reindex(pd.MultiIndex.from_frame(rows[['patient_id','ICULOS']])).to_numpy())
    checks.append(dict(check='saved_a_test_auc_and_original_labels',artifact='a_test_initial.csv',passed=True,rows=len(rows)))
    assert actions.state_end_day.eq(actions.day-1).all()
    assert (actions.training_max_day<actions.day).all()
    assert (actions.training_label_available_max<=actions.day).all()
    assert (actions.label_received_day==actions.day+1).all()
    for name in ACTIONS:
        assert actions.loc[actions.executed.eq(ACTIONS.index(name)),f'mask_{name}'].all()
    assert daily.loc[daily.warmup,'action'].eq('A1').all()
    assert daily.groupby(['scenario','policy']).label_received_day.max().eq(DAYS+1).all()
    checks.extend([dict(check=name,artifact='daily/actions',passed=True,rows=len(actions)) for name in
                   ['past_only_state_and_fits','next_day_reward_alignment','action_masks','warmup_and_final_release']])
    return table('integrity_checks',checks)

## Ejecución completa con persistencia parcial

In [ ]:
def run_experiment():
    """Run the fixed A protocol, B-only training and frozen B-test assessment."""
    checkpoint('setup')
    frame,partitions,first_positive=load_frame()
    a_replays={name:make_replay(frame,partitions[name],A_DAYS,BEDS) for name in ('a_train','a_validation','a_test')}
    b_replays={name:make_replay(frame,partitions[name]) for name in ('b_train','b_validation','b_test')}
    rng=np.random.default_rng(SEED)
    train_bases=[make_replay(frame,partitions['b_train'],rng=rng) for _ in range(4)]
    manifest=[];counts=[]
    for name,ids in partitions.items():
        manifest.extend(dict(partition=name,patient_id=pid,ever_sepsis=pid in first_positive) for pid in ids)
        r=(a_replays|b_replays)[name]
        counts.append(dict(partition=name,patients_available=len(ids),patients_used=r.patient_id.nunique(),
                           rows_consumed=len(r),days=int(r.day.max()),beds=BEDS,capacity_passed=True))
    table('patient_partitions',manifest);table('partition_counts',counts)
    for i,r in enumerate(train_bases):
        counts.append(dict(partition=f'b_train_replay_{i}',patients_available=len(partitions['b_train']),
                           patients_used=r.patient_id.nunique(),rows_consumed=len(r),days=DAYS,beds=BEDS,capacity_passed=True))
    table('partition_counts',counts)
    for name,r in a_replays.items():
        a_replays[name]=inject(r,'stable',0.,0.,ONSET,np.zeros(7),np.ones(7),first_positive)
    checkpoint('capacity_validated')
    initial,auc_reference=prepare_initial(a_replays)
    cov_rate,concept_rate,cost_unit,calibration=calibrate(b_replays['b_train'],initial,auc_reference,first_positive)
    agent,history,training=train_agent(train_bases,b_replays['b_validation'],initial,auc_reference,cost_unit,cov_rate,concept_rate,first_positive)
    frozen=agent.fingerprint();all_daily=[];all_actions=[];all_comparisons=[];all_audits=[];calendar=[]
    for scenario in SCENARIOS:
        schedule=inject(b_replays['b_test'],scenario,cov_rate,concept_rate,ONSET,initial.med,initial.iqr,first_positive)
        assert np.array_equal(schedule.SepsisLabel,b_replays['b_test'].SepsisLabel)
        calendar.extend(schedule.groupby('day')[['cov_level','concept_shift_hours']].first().reset_index().assign(scenario=scenario).to_dict('records'))
        for policy in ('ppo',*ACTIONS,'random','heuristic'):
            result,actions,daily,audits=roll(schedule,policy,initial,auc_reference,cost_unit,agent,audit=policy=='ppo',export=True)
            all_comparisons.append(result);all_daily.append(daily);all_actions.append(actions)
            if len(audits):all_audits.append(audits)
            table('policy_comparisons',all_comparisons);table('daily_metrics',pd.concat(all_daily,ignore_index=True))
            table('actions',pd.concat(all_actions,ignore_index=True))
            if all_audits:table('oracle_audit',pd.concat(all_audits,ignore_index=True))
            checkpoint('testing',completed_scenario=scenario,completed_policy=policy)
        checkpoint('scenario_evaluated',completed_scenario=scenario)
    daily=pd.concat(all_daily,ignore_index=True);actions=pd.concat(all_actions,ignore_index=True)
    comparisons=pd.DataFrame(all_comparisons);audits=pd.concat(all_audits,ignore_index=True)
    table('drift_calendar',calendar)
    assert agent.fingerprint()==frozen
    checks=verify_exports(daily,actions,initial,agent,frame)
    table('controller_freeze',[dict(validation_and_test_frozen=True,selected_weights_sha256=frozen,oracle_training_transitions=0)])
    render_figures(a_replays,b_replays,history,daily,actions,comparisons,audits)
    checkpoint('complete',status='complete',integrity_passed=bool(checks.passed.all()),
               predictions_files=len(list((OUT/'predictions').glob('*.csv'))),figure_files=len(list((OUT/'figures').glob('*.png'))),
               auc_reference=auc_reference,cov_rate=cov_rate,concept_rate=concept_rate,cost_unit=cost_unit,
               selected_weights_sha256=frozen)
    return comparisons

## execute

In [ ]:
try:
    results=run_experiment()
    print(results.to_string(index=False))
except Exception as error:
    checkpoint('failed',status='failed',error_type=type(error).__name__,error=str(error),traceback=traceback.format_exc())
    print('Experiment failed; inspect run_summary.csv. Partial CSV outputs are retained.',flush=True)